# Phase 2 -- Data Pipeline

Alert Intelligence Engine -- master plan Phase 2 (section 20): "Ingestion, validation, normalization, versioning." Gate: data tests pass.

This notebook demonstrates the pipeline built in `pipelines/validation/schema_registry.py`, `pipelines/normalization/*`, and `pipelines/ingestion/identifiers.py`. It does not re-derive the logic inline -- it calls the same modules `tests/` exercise, so what you see running here is exactly what's under test (61/61 passing, see final cell).

**What this phase adds over Phase 1 (audit-only):**
- Schema registry with per-column semantic type (id/date/numeric/categorical/text/leakage) and validation
- Strict type normalization -- every transform is additive (`(Normalized)`/`(Parsed)` companion columns), raw values are never overwritten except two documented storage-safety casts (see below)
- Country/nationality normalization grounded in the actual value patterns observed in this workbook
- A dedicated classifier for `Hit Details (DOB)` -- a field that looks like a date column but is not (see finding below)
- Stable, namespaced customer/transaction/record identifiers
- Near-duplicate candidate detection (exact-match on normalized identity + event time, not a fuzzy/heuristic engine)
- Explicit missingness indicators for partially-missing fields
- A feature-availability timestamp rule for later temporal experiments
- Dataset + schema versioning, with the normalized data persisted locally (never committed -- still real PII)

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import json
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Schema validation

In [2]:
from pipelines.ingestion.load_alerts import load_raw_alerts
from pipelines.validation.schema_registry import validate_all

raw_sheets = load_raw_alerts()
validation_results = validate_all(raw_sheets)

for name, result in validation_results.items():
    status = "PASS" if result.passed else "FAIL"
    print(f"{status}: {name} -- missing_required={result.missing_required_columns}, "
          f"unexpected={result.unexpected_columns}")

assert all(r.passed for r in validation_results.values())

PASS: CustomerViolation -- missing_required=[], unexpected=[]
PASS: TransactionNameViolation -- missing_required=[], unexpected=[]
PASS: Rule -- missing_required=[], unexpected=[]


## 2. A finding this phase surfaced: `Hit Details (DOB)` is not a clean date field

Discovered while building the pipeline, not assumed in advance: this column mixes real dates, bare years, garbage integers, multi-value lists, date ranges and "circa" qualifiers. Blindly parsing it with `pd.to_datetime` would have silently turned a bare year like `1965` into an ordinal-timestamp date -- wrong data that looks valid. `pipelines/normalization/dob_text.py` classifies instead of guessing.

In [3]:
from pipelines.normalization.dob_text import classify_hit_dob

sample = raw_sheets["CustomerViolation"]["Hit Details (DOB)"]
print("Distinct native Python types in this column:",
      sample.dropna().map(type).value_counts().to_dict())

classified = classify_hit_dob(sample)
print(f"\nResolved to a full date: {classified['parsed_date'].notna().sum()}")
print(f"Resolved to year-only:   {(classified['year'].notna() & classified['parsed_date'].isna()).sum()}")
print(f"Flagged multi-value:     {classified['is_multi_value'].sum()}")
print(f"Flagged unresolved:      {classified['is_unresolved'].sum()}")

print("\nExamples of unresolved values (never guessed into a date):")
unresolved_examples = sample[classified['is_unresolved']].dropna().unique()[:8]
for v in unresolved_examples:
    print(" -", repr(v))

Distinct native Python types in this column: {<class 'datetime.datetime'>: 718, <class 'int'>: 517, <class 'str'>: 437}

Resolved to a full date: 720
Resolved to year-only:   516
Flagged multi-value:     426
Flagged unresolved:      10

Examples of unresolved values (never guessed into a date):
 - '1957/02/00'
 - '1975/10/00'
 - '1960/02/00'
 - '1974/04/00'
 - 1033879
 - '1978/09/00'
 - '1978/11/00'
 - '1958/00/00'


## 3. Country/nationality normalization

Grounded in the actual patterns in this workbook: `"Alerted Party Nationality"` columns are always `"XX-COUNTRY NAME"`; `"Hit Details (Nationality)"` columns are far messier (bare codes, bare names, pipe/semicolon multi-value lists, and values that look truncated at a fixed length).

In [4]:
from pipelines.normalization.geo import normalize_country_field

raw_col = raw_sheets["CustomerViolation"]["Alerted Party Nationality"]
result = normalize_country_field(raw_col)
print("Alerted Party Nationality -- before -> after (first 5 distinct):")
for raw, norm in list(dict(zip(raw_col, result["normalized"])).items())[:5]:
    print(f"  {raw!r:35} -> {norm!r}")

raw_col2 = raw_sheets["CustomerViolation"]["Hit Details (Nationality)"]
result2 = normalize_country_field(raw_col2)
n_multi = result2["is_multi_value"].sum()
n_unresolved = result2["unresolved_token_present"].sum()
print(f"\nHit Details (Nationality): {n_multi} multi-value cells, "
      f"{n_unresolved} flagged as containing a likely-truncated fragment (not repaired).")

Alerted Party Nationality -- before -> after (first 5 distinct):
  'LB-LEBANON'                        -> 'LEBANON'
  'AE-UNITED ARAB EMIRATES'           -> 'UNITED ARAB EMIRATES'
  'IN-INDIA'                          -> 'INDIA'
  'UG-UGANDA'                         -> 'UGANDA'
  'EG-EGYPT'                          -> 'EGYPT'

Hit Details (Nationality): 36 multi-value cells, 13 flagged as containing a likely-truncated fragment (not repaired).


## 4. Run the full normalization pipeline for all three sheets

In [5]:
from pipelines.normalization.pipeline import run_phase2_pipeline

normalized_sheets, quality_report = run_phase2_pipeline(REPO_ROOT, persist=True)

print("Overall status:", quality_report["overall_status"])
print("Dataset version:", quality_report["dataset_version"])
print("Schema version:", quality_report["schema_version"])

Overall status: PASS
Dataset version: v1-46641a47
Schema version: 1.0.0


In [6]:
for name, df in normalized_sheets.items():
    raw_cols = quality_report["sheets"][name]["raw_cols"]
    print(f"{name}: {raw_cols} raw columns -> {len(df.columns)} columns after normalization "
          f"(+{len(df.columns) - raw_cols} derived)")

print()
print(normalized_sheets["CustomerViolation"][[
    "UIN", "customer_id", "record_id",
    "Alerted Party Name", "Alerted Party Name (Normalized)",
    "Alerted Party Nationality", "Alerted Party Nationality (Normalized)",
]].head(5))

CustomerViolation: 23 raw columns -> 54 columns after normalization (+31 derived)
TransactionNameViolation: 25 raw columns -> 60 columns after normalization (+35 derived)
Rule: 24 raw columns -> 56 columns after normalization (+32 derived)

       UIN                         customer_id         record_id                          Alerted Party Name             Alerted Party Name (Normalized)  \
0  1502141  [REDACTED-ID]  564e19f933e10b2e                               [REDACTED-PII]                               [REDACTED-PII]   
1  1502244  [REDACTED-ID]  d35684530fa6209a                       [REDACTED-PII]                       [REDACTED-PII]   
2  1503196  [REDACTED-ID]  6f18c578fed9486b  [REDACTED-PII]  [REDACTED-PII]   
3  1503585  [REDACTED-ID]  e1bfb22617e5300f                           [REDACTED-PII]                           [REDACTED-PII]   
4  1503585  [REDACTED-ID]  40f4bd14f29637a7                           [REDACTED-PII]                           [REDACTED-PII]   

  Alert

## 5. Duplicate and near-duplicate findings

Exact duplicates match Phase 1 exactly (unchanged data, different tooling -- confirms both audits agree). Near-duplicate candidates use exact-match grouping on normalized identity + alert timestamp (or transaction reference + rule name for Rule) -- never a fuzzy/similarity score, per the master plan's no-heuristic-matching-engine rule. These are *candidates for review*, not an automatic dedup decision.

In [7]:
for name, s in quality_report["sheets"].items():
    print(f"{name}: exact_duplicate_rows={s['exact_duplicate_rows']}, "
          f"near_duplicate_candidate_rows={s['near_duplicate_candidate_rows']}")

# Phase 1 cross-check
phase1_report = json.load(open(REPO_ROOT / "evaluation" / "phase1_data_audit_report.json"))
for name in quality_report["sheets"]:
    p1 = phase1_report["duplicate_rows"][name]
    p2 = quality_report["sheets"][name]["exact_duplicate_rows"]
    assert p1 == p2, f"Phase 1/Phase 2 duplicate count mismatch for {name}: {p1} vs {p2}"
print("\nExact-duplicate counts agree with Phase 1 audit for all sheets.")

CustomerViolation: exact_duplicate_rows=8, near_duplicate_candidate_rows=458
TransactionNameViolation: exact_duplicate_rows=537, near_duplicate_candidate_rows=421
Rule: exact_duplicate_rows=0, near_duplicate_candidate_rows=756

Exact-duplicate counts agree with Phase 1 audit for all sheets.


## 6. Storage-safety casts (documented, not silent)

In [8]:
print("Columns cast to their str() display value for parquet storage")
print("(raw content preserved exactly as text -- see types.stringify_for_storage docstring):\n")
for name, cols in quality_report["storage_stringified_columns"].items():
    print(f"{name}: {cols if cols else '(none needed)'}")

Columns cast to their str() display value for parquet storage
(raw content preserved exactly as text -- see types.stringify_for_storage docstring):

CustomerViolation: (none needed)
TransactionNameViolation: ['Alerted Party DOB']
Rule: ['Beneficiary Id Number', 'Beneficiary Relationship']


## 7. Leakage registry enforcement check

In [9]:
from pipelines.validation.temporal import assert_no_post_scoring_columns, POST_SCORING_COLUMNS

for name, df in normalized_sheets.items():
    leakage_cols = POST_SCORING_COLUMNS[name]
    derived_from_leakage = [
        c for c in df.columns
        if any(c.startswith(f"{lc} (") for lc in leakage_cols)
    ]
    print(f"{name}: leakage columns present={[c for c in leakage_cols if c in df.columns]}, "
          f"derived features from leakage columns={derived_from_leakage}")
    assert derived_from_leakage == [], "Leakage field must never get a derived feature column"

# Sanity check the guard itself raises when it should
try:
    assert_no_post_scoring_columns(["Maker Comment"], "CustomerViolation")
    raise AssertionError("Expected ValueError, guard did not raise")
except ValueError as e:
    print("\nGuard correctly raises on a leakage column in a feature list:")
    print(" ", e)

CustomerViolation: leakage columns present=['Maker Name', 'Maker Comment Date', 'Maker Comment', 'Alert Closure Date & Time', 'Alert Status'], derived features from leakage columns=[]
TransactionNameViolation: leakage columns present=['Maker Name', 'Maker Comment Date', 'Maker Comment', 'Alert Status'], derived features from leakage columns=[]
Rule: leakage columns present=['Comment', 'Actiondate', 'Action Taken By', 'Status'], derived features from leakage columns=[]

Guard correctly raises on a leakage column in a feature list:
  Post-scoring-time (leakage) columns present in feature set for 'CustomerViolation': ['Maker Comment']. These are only known after human review and must never be live model input.


## 8. Persisted artifacts

In [10]:
version_dir = REPO_ROOT / "data" / "normalized" / quality_report["dataset_version"]
print("Persisted to (gitignored, local-only):", version_dir)
for p in sorted(version_dir.iterdir()):
    print(" -", p.name, f"({p.stat().st_size:,} bytes)")

# committed, non-PII copy for the repo
eval_out = REPO_ROOT / "evaluation" / "phase2_data_pipeline_report.json"
with open(eval_out, "w") as f:
    json.dump(quality_report, f, indent=2, default=str)
print("\nCommitted aggregate-only report:", eval_out)

Persisted to (gitignored, local-only): /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/data/normalized/v1-46641a47
 - CustomerViolation.parquet (426,521 bytes)
 - Rule.parquet (729,467 bytes)
 - TransactionNameViolation.parquet (226,305 bytes)
 - manifest.json (2,026 bytes)

Committed aggregate-only report: /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/evaluation/phase2_data_pipeline_report.json


## 9. Test suite

All logic demonstrated above is covered by `tests/` -- run here for the record.

In [11]:
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-q"],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
assert result.returncode == 0, "Test suite must pass before Phase 2 is considered done" 

.............................................................            [100%]
61 passed in 4.69s



## Phase 2 -- Result

**Status: PASS**

- Schema registry validates all three sheets; matches Phase 1's schema findings exactly.
- Type normalization is additive-only (raw columns preserved) except two documented storage-safety string casts on genuinely mixed-type source columns.
- `Hit Details (DOB)` identified as a heterogeneous free-text field masquerading as a date column -- classified, not guessed; unresolved/multi-value cases flagged for client clarification rather than silently parsed.
- Country/nationality normalization grounded in observed patterns; unresolved (likely-truncated) fragments flagged, not repaired.
- Stable, sheet-namespaced customer/transaction/record identifiers added -- explicitly documented as NOT a proven cross-sheet identifier space.
- Near-duplicate candidates detected via exact-match grouping only (no fuzzy/heuristic matching engine, per master plan rule).
- Explicit missingness indicators added only for partially-missing fields (POB, 100% missing, deliberately excluded).
- Leakage columns get zero derived feature columns -- verified by test.
- Dataset persisted with a content-hash-derived version tag; normalized data never committed (PII).
- 61/61 unit + integration tests passing.

**Next gate:** Phase 3 -- entity feature/representation building (Customer Name + Transaction Name alerts). Awaiting human approval to proceed.